# Deep Learning with MNIST: From CNNs to Conditional Latent Diffusion

In this notebook you will build, step by step, the components of a **conditional latent diffusion model**.  
Each task reuses modules from the previous one, so no pressure, but if you mess up Task 1 everything else collapses. Good luck.

**Roadmap:**
1. **Task 1** — CNN Dimension Reduction Block + Classifier  
2. **Task 2** — Convolutional Autoencoder (reusing the encoder from Task 1)  
3. **Task 3** — Transformer Layer with Cross-Attention  
4. **Task 4** — Conditional Latent Diffusion Model (combining Tasks 2 & 3)

## Setup
Imports and MNIST data loading. Nothing to implement here — consider it a gift.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# -- Hyperparameters that you will reference throughout the notebook --
BATCH_SIZE = 128
NUM_CLASSES = 10          # MNIST has 10 digits, shocking
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -- MNIST data: because every ML journey begins here, whether you like it or not --
transform = transforms.Compose([
    transforms.ToTensor(),           # [0,1] range, (1,28,28)
])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"Training samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
print(f"Device: {DEVICE}, {'nice GPU you got there' if DEVICE.type == 'cuda' else 'CPU it is, grab a coffee'}")

---
## Task 1: CNN Dimension Reduction Block & Classifier (TODO)

### 1.1 — `DimensionReductionBlock`

Build a convolutional encoder that maps an input image to a flat feature vector.
This block will be reused as the encoder in the autoencoder later. 

In [ ]:
class DimensionReductionBlock(nn.Module):
    """
    Convolutional encoder: (B, in_channels, 28, 28) -> (B, output_dim).
    """

    def __init__(self, in_channels=1, output_dim=64):
        super().__init__()

        # Create a convolutional network of which the output dimensions will have
        # 128 channels and an image size of 4x4
        # Following classes are useful:
        # nn.Conv2d(in_channels, out_channels, kernel_size=k, stride=s, padding=p)
        # nn.MaxPool2d(kernel_size)
        # nn.ReLU() as the non-linearity
        # I would recommend to have three convolutional layers, where the number of channels increases with each. 
        # The sequential class takes all the building blocks you enter and self.conv(x) will then execute them. 
        self.conv = nn.Sequential(
             ...
        )
        self.fc = nn.Linear(128 * 4 * 4, output_dim)

    def forward(self, x):
        # x: (B, in_channels, 28, 28)
        batch_size = x.size(0)
        x = self.conv(x)                    # (B, 128, 4, 4)
        # The last linear layer always operates on the last dimension. Use x.view() to reshape the tensor to (B, 2048)
        x = ...
        z = self.fc(x)                       # (B, output_dim)
        return z

### 1.2 — `CNNClassifier`

Here we wrap`DimensionReductionBlock` with a linear classification head.  
The `output_dim` of the reduction block is the input to the head.

In [ ]:
class CNNClassifier(nn.Module):
    """
    Slap a linear head on the encoder. State of the art circa 2014.
    """

    def __init__(self, output_dim=64, num_classes=NUM_CLASSES):
        super().__init__()
        self.encoder = DimensionReductionBlock(in_channels=1, output_dim=output_dim)
        self.head = nn.Linear(output_dim, num_classes)

    def forward(self, x):
        z = self.encoder(x)          # (B, output_dim)
        logits = self.head(z)         # (B, num_classes)
        return logits

### 1.3 — Train the Classifier

In [ ]:
# -- Training the classifier --
CLASSIFIER_EPOCHS = 3

classifier = CNNClassifier(output_dim=64, num_classes=NUM_CLASSES).to(DEVICE)
optimizer_cls = torch.optim.Adam(classifier.parameters(), lr=1e-3)

for epoch in range(1, CLASSIFIER_EPOCHS + 1):
    classifier.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        logits = classifier(images)
        loss = F.cross_entropy(logits, labels)

        optimizer_cls.zero_grad()
        loss.backward()
        optimizer_cls.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += images.size(0)

    print(f"Epoch {epoch}/{CLASSIFIER_EPOCHS}  |  "
          f"Loss: {total_loss / total:.4f}  |  "
          f"Acc: {100 * correct / total:.2f}%")

# -- Quick eval --
classifier.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (classifier(images).argmax(1) == labels).sum().item()
        total += images.size(0)
print(f"\nTest accuracy: {100 * correct / total:.2f}% — not bad for three epochs of MNIST")

---
## Task 2: Convolutional Autoencoder (TODO)

### 2.1 — `CNNDecoder`

Here we build a decoder that mirrors the encoder: it takes a latent vector `(B, latent_dim)` and reconstructs an image `(B, 1, 28, 28)`.
We will apply transposed convolution. To get an intuition on how this operates, check out: https://github.com/vdumoulin/conv_arithmetic/blob/master/README.md
The `output_padding` values are not arbitrary. They ensure exact dimensional recovery. 

In [ ]:
class CNNDecoder(nn.Module):
    """
    Mirrors the DimensionReductionBlock. (B, latent_dim) -> (B, 1, 28, 28).
    ConvTranspose2d: where dreams of clean math go to die.
    """

    def __init__(self, latent_dim=64):
        super().__init__()

        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)
        
        # Create a convolutional network of which the output dimensions will have
        # 1 channels and an image size of 28x28
        # Following classes are useful:
        # nn.ConvTranspose2d(in_channels, out_channels, kernel_size=k, stride=s, padding=p, output_padding=p_out)
        # nn.Sigmoid()
        # nn.ReLU()
        # I would recommend to have three deconvolutional layers, where the number of channels decreases with each. 
        self.deconv = nn.Sequential(
            ...
        )

    def forward(self, z):
        # z: (B, latent_dim)
        x = self.fc(z)                      # (B, 128*4*4)
        x = x.view(-1, 128, 4, 4)           # (B, 128, 4, 4) — unflatten
        x_hat = self.deconv(x)              # (B, 1, 28, 28)
        return x_hat

### 2.2 — `Autoencoder` (TODO)

Combine the `DimensionReductionBlock` (encoder) and `CNNDecoder` into a single autoencoder.  
The `forward` pass returns both the reconstruction and the latent vector (you will need the latents later in Task 4).

In [ ]:
class Autoencoder(nn.Module):
    """
    Encoder + Decoder = Autoencoder. Revolutionary naming conventions.
    Returns (reconstruction, latent) because Task 4 will need those latents.
    """

    def __init__(self, latent_dim=64):
        super().__init__()
        self.encoder = DimensionReductionBlock(in_channels=1, output_dim=latent_dim)
        self.decoder = CNNDecoder(latent_dim=latent_dim)

    def forward(self, x):
        # Take the image as the input, and return the reconstruction and latent encoding (in this order), eg return x_hat, z
        z = ...              # (B, latent_dim)
        x_hat = ...          # (B, 1, 28, 28)
        return x_hat, z

### 2.3 — Train the Autoencoder (TODO)

In [ ]:
# -- Training the autoencoder --
AE_EPOCHS = 10
# How low can you go in the latent dimension? 
LATENT_DIM = ...
autoencoder = Autoencoder(latent_dim=LATENT_DIM).to(DEVICE)
optimizer_ae = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)

for epoch in range(1, AE_EPOCHS + 1):
    autoencoder.train()
    total_loss = 0.0

    for images, _ in train_loader:
        images = images.to(DEVICE)

        x_hat, z = autoencoder(images)
        loss = F.mse_loss(x_hat, images)     # pixel-wise reconstruction loss

        optimizer_ae.zero_grad()
        loss.backward()
        optimizer_ae.step()

        total_loss += loss.item() * images.size(0)

    avg_loss = total_loss / len(train_dataset)
    print(f"Epoch {epoch}/{AE_EPOCHS}  |  Recon Loss: {avg_loss:.6f}")

print("Autoencoder training complete.")

### 2.4 — Visualise Reconstructions

In [ ]:
# -- Visualise some reconstructions to confirm the autoencoder isn't just outputting grey blobs --
autoencoder.eval()
with torch.no_grad():
    sample_images, _ = next(iter(test_loader))
    sample_images = sample_images[:8].to(DEVICE)
    recon, _ = autoencoder(sample_images)

fig, axes = plt.subplots(2, 8, figsize=(14, 5))
for i in range(8):
    axes[0, i].imshow(sample_images[i].cpu().squeeze(), cmap="gray")
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].cpu().squeeze(), cmap="gray")
    axes[1, i].set_title("Recon")
    axes[1, i].axis("off")
plt.suptitle("Autoencoder Reconstructions. If these look identical you're either great or overfitting")
plt.tight_layout()
plt.show()

---
## Task 3: Transformer Layer with Cross-Attention (TODO)

Implement a single Transformer layer following the architecture from *Attention Is All You Need* (Vaswani et al., 2017).

Your layer must support **cross-attention**: when a `context` tensor is provided. 
When no context is provided, the layer performs standard self-attention only.
This is the component you were taught for self-attention in class, the cross-attention variant is your extrapolation exercise.

In [ ]:
class TransformerLayer(nn.Module):
    """
    One Transformer layer: self-attention + optional cross-attention + FFN.

    Args:
        input_dim: input dimension
        d_model:   embedding dimension
        n_heads:   number of attention heads (must divide d_model evenly, obviously)
        d_ff:      feed-forward hidden dimension
        dropout:   dropout rate
    """

    def __init__(self, input_dim, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads. Basic arithmetic."

        self.d_model = d_model # In this dimension, we do the self & cross attention
        self.n_heads = n_heads # This is the number of heads. d_model must be divisible by this number
        self.d_k = d_model // n_heads     # per-head dimension
        
        # Self-attention projections (Q, K, V, and output — all by hand)
        self.norm1 = nn.LayerNorm(input_dim)
        # Use nn.Linear to define the attention projections
        self.W_q_sa = ...
        self.W_k_sa = ...
        self.W_v_sa = ...
        self.W_o_sa = ... # (This one maps back to the input dimension)

        # We want to condition on some input. It is common to use a second attention layer for cross attention
        # Hence we have to also define these projections:
        # Cross-attention projections (same idea, separate weights)
        self.norm2 = nn.LayerNorm(input_dim)
        self.W_q_ca = ...
        self.W_k_ca = ...
        self.W_v_ca = ...
        self.W_o_ca = ...

        # This is the MLP after the attention step
        # Position-wise feed-forward network
        self.norm3 = nn.LayerNorm(input_dim)
        self.ff = nn.Sequential(
            nn.Linear(input_dim, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, input_dim),
        )

        self.dropout = nn.Dropout(dropout)

    def _multihead_attention(self, q, k, v, W_q, W_k, W_v, W_o):
        """
        Scaled dot-product multi-head attention.
        No magic, just matrix multiplications and a softmax. 

        Args:
            q: (B, S_q, input_dim) — queries
            k: (B, S_k, input_dim) — keys
            v: (B, S_k, input_dim) — values (same sequence length as keys)
            W_q, W_k, W_v: projection layers (input_dim -> d_model)
            W_o: output projection layer (d_model -> input_dim)

        Returns:
            (B, S_q, d_model) — attended output
        """
        B, S_q, _ = q.shape
        S_k = k.shape[1]

        # Step 1: project Q, K, V through their respective linear layers
        Q = ...   # (B, S_q, d_model)
        K = ...    # (B, S_k, d_model)
        V = ...   # (B, S_k, d_model)

        # Here we reshape the projections such that we can compute the attentions in parallel for each head. 
        # Matrix multiplication operates on the last two dimensions. 
        Q = Q.view(B, S_q, self.n_heads, self.d_k).transpose(1, 2)   # (B, n_heads, S_q, d_k)
        K = K.view(B, S_k, self.n_heads, self.d_k).transpose(1, 2)   # (B, n_heads, S_k, d_k)
        V = V.view(B, S_k, self.n_heads, self.d_k).transpose(1, 2)   # (B, n_heads, S_k, d_k)

        # Step 3: scaled dot-product attention
        # The following functions are suggested for usage:
        # torch.matmul(A,B) calculates the matrix multiplication on the last two dimensions in parallel. 
        # .transpose(-2,-1) transposes the last two dimensions.
        # F.softmax(input, dim=dimension) calculates the softmax along your desired dimension 
        scores = ...  # (B, n_heads, S_q, S_k)
        attn_weights = ...                            # (B, n_heads, S_q, S_k)
        attn_weights = self.dropout(attn_weights)                           # dropout on attention — yes, really
        attn_out = ...                           # (B, n_heads, S_q, d_k)

        # Here we concatenate heads and project back
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S_q, self.d_model)  # (B, S_q, d_model)
        output = W_o(attn_out)                                              # (B, S_q, input_dim)
        return output

    def forward(self, x, context=None):
        """
        Args:
            x:       (B, S, d_model) — the main sequence
            context: (B, T, d_model) — optional conditioning sequence for cross-attention.
                     If None, cross-attention is skipped entirely.
        Returns:
            (B, S, d_model)
        """
        # --- Self-Attention: Q, K, V all come from x ---
        x_norm = self.norm1(x)
        # What are the inputs for self attention? 
        sa_out = self._multihead_attention(
            q=..., k=..., v=...,
            W_q=self.W_q_sa, W_k=self.W_k_sa, W_v=self.W_v_sa, W_o=self.W_o_sa,
        )
        x = x + self.dropout(sa_out)

        if context is not None:
            # What are the inputs for cross attention?
            # Remember, I want to condition on my context. 
            # Hint: What dimensions do you need as the output?
            x_norm = self.norm2(x)
            ca_out = self._multihead_attention(
                q=..., k=..., v=...,
                W_q=self.W_q_ca, W_k=self.W_k_ca, W_v=self.W_v_ca, W_o=self.W_o_ca,
            )
            x = x + self.dropout(ca_out)

        # --- Feed-Forward ---
        x_norm = self.norm3(x)
        ff_out = self.ff(x_norm)
        x = x + self.dropout(ff_out)

        return x

### 3.1 — Sanity Check
Verify that the transformer layer works for both self-attention and cross-attention.

In [ ]:
# -- Quick sanity check: does it run without exploding? --
emb_dim, d_model, n_heads, d_ff = 64, 128, 4, 256

layer = TransformerLayer(emb_dim, d_model, n_heads, d_ff).to(DEVICE)

# Self-attention only
x_dummy = torch.randn(4, 10, emb_dim, device=DEVICE)          # (B=4, S=10, d=64)
out_sa = layer(x_dummy)
print(f"Self-attention only  — input: {x_dummy.shape}, output: {out_sa.shape}")

# With cross-attention
ctx_dummy = torch.randn(4, 5, emb_dim, device=DEVICE)         # (B=4, T=5, d=64)
out_ca = layer(x_dummy, context=ctx_dummy)
print(f"With cross-attention — input: {x_dummy.shape}, context: {ctx_dummy.shape}, output: {out_ca.shape}")

assert out_sa.shape == x_dummy.shape, "Output shape mismatch (self-attn)"
assert out_ca.shape == x_dummy.shape, "Output shape mismatch (cross-attn)"
print("\nAll shapes check out. The transformer layer lives another day.")

---
## Task 4: Conditional Latent Diffusion Model (TODO)

Now combine everything. You will build a **conditional latent diffusion model** that:

1. Uses the **pretrained autoencoder** (Task 2) to map images into a latent space.  
2. Performs the **forward (noising) and reverse (denoising) diffusion process** in that latent space.  
3. Uses a stack of **Transformer layers with cross-attention** (Task 3) as the denoising network, where the conditioning signal (class label) is injected via cross-attention.

In [ ]:
class ConditionalLatentDiffusionModel(nn.Module):
    """
    Latent diffusion with transformer-based denoising and class-conditional cross-attention.
    """

    def __init__(
        self,
        autoencoder,
        latent_dim=64,
        emb_dim=128,
        d_model=128,
        n_heads=4,
        d_ff=256,
        num_classes=NUM_CLASSES,
        num_timesteps=200,
        num_layers=4,
        dropout=0.1,
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.emb_dim = emb_dim # We work in an embedded dimension. We take a latent scalar and map it to this dimension
        self.num_timesteps = num_timesteps # How many noise steps are modeled

        # Freeze the pretrained autoencoder, we trained it already before
        self.autoencoder = autoencoder
        for p in self.autoencoder.parameters():
            p.requires_grad = False

        # --- Per-token projection: each scalar in the latent vector -> emb_dim ---
        self.token_proj = nn.Linear(1, emb_dim)

        # --- Positional embedding: one learnable vector per latent index ---
        # Tells the transformer "you are latent dimension #k"
        self.pos_embed = nn.Embedding(latent_dim, emb_dim)

        # --- Time embedding: shared across all tokens ---
        # Tells the transformer "we are at diffusion step t"
        self.time_embed = nn.Sequential(
            nn.Embedding(num_timesteps, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim),
        )

        # --- Class embedding: injected via cross-attention ---
        self.class_embed = nn.Embedding(num_classes, emb_dim)

        # --- Transformer denoising backbone ---
        self.transformer_layers = nn.ModuleList([
            TransformerLayer(emb_dim, d_model, n_heads, d_ff, dropout=dropout)
            for _ in range(num_layers)
        ])

        # --- Project each token back to a single scalar (predict noise per dimension) ---
        self.output_proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 1),
        )

        # --- Linear noise schedule ---
        betas = torch.linspace(1e-4, 0.02, num_timesteps)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)

        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cumprod", alphas_cumprod)

    def _embed_latent(self, z_t, t):
        """
        Turn a flat latent vector into a sequence of embedded tokens
        with positional and temporal information baked in.

        z_t: (B, latent_dim) -> tokens: (B, latent_dim, emb_dim)
        """
        B = z_t.size(0)

        # Each scalar becomes a token: (B, latent_dim) -> (B, latent_dim, 1) -> (B, latent_dim, emb_dim)
        tokens = self.token_proj(z_t.unsqueeze(-1))

        # Positional embedding: same for every sample in the batch
        positions = torch.arange(self.latent_dim, device=z_t.device)   # (latent_dim,)
        pos_emb = self.pos_embed(positions)                             # (latent_dim, emb_dim)

        # Time embedding: same across all tokens in a sample
        t_emb = self.time_embed(t)                                      # (B, emb_dim)

        # Combine: token value + where it lives + when we are
        tokens = tokens + pos_emb.unsqueeze(0) + t_emb.unsqueeze(1)

        return tokens

    # ---- Forward diffusion (noising) ----
    def forward_diffusion(self, z_0, t):
        """
        q(z_t | z_0): add noise to clean latent at timestep t.
        Returns (z_t, epsilon). 
        """
        alpha_bar_t = self.alphas_cumprod[t].unsqueeze(-1)    # (B, 1)
        epsilon = torch.randn_like(z_0)                        # (B, latent_dim)

        # Create the noisy latent vector
        z_t = ...
        return z_t, epsilon

    # ---- Noise prediction network ----
    def predict_noise(self, z_t, t, class_labels):
        """
        Takes noisy latent z_t, timestep t, class label -> predicts epsilon that was added to z_{t-1}.
        """
        # Embed the noisy latent and the class labels into a sequence suitable for a transformer
        x = self._embed_latent(z_t, t)                        # (B, latent_dim, emb_dim)

        context = self.class_embed(class_labels).unsqueeze(1)  # (B, 1, emb_dim)

        # Let the transformer layers do their thing
        for layer in self.transformer_layers:
            x = layer(x, context=context)

        # Project each token back to a scalar -> reconstruct the noise vector
        noise_pred = self.output_proj(x).squeeze(-1)           # (B, latent_dim)
        return noise_pred

    # ---- Training step ----
    def compute_loss(self, images, labels):
        """
        Encode -> noise -> predict noise -> MSE.
        """
        with torch.no_grad():
            z_0 = self.autoencoder.encoder(images)
        
        B = z_0.size(0)

        # Sample noise time steps for each sample. 
        # Apply forward diffusion with forward_diffusion
        # Predict the added noise epsilon
        # Use torch.randint(lowest_val, highest_val, size, device=z_0.device)
        # and the class functions forward_diffusion and predict_noise
        # As we aim to predict a gaussian noise, we can use the function F.mse_loss(prediction, target)
        t = ...

        
        return loss

    # ---- Sampling (reverse diffusion) ----
    @torch.no_grad()
    def sample(self, class_labels, device=None):
        """
        DDPM reverse process. Still slow. Still works.
        """
        device = device or next(self.parameters()).device
        class_labels = class_labels.to(device)
        B = class_labels.size(0)

        z = torch.randn(B, self.latent_dim, device=device)

        for t in reversed(range(self.num_timesteps)):
            t_batch = torch.full((B,), t, device=device, dtype=torch.long)
            eps_pred = self.predict_noise(z, t_batch, class_labels)

            alpha_t = self.alphas[t]
            alpha_bar_t = self.alphas_cumprod[t]

            z = (1.0 / torch.sqrt(alpha_t)) * (
                z - (self.betas[t] / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred
            )

            if t > 0:
                z = z + torch.sqrt(self.betas[t]) * torch.randn_like(z)

        images = self.autoencoder.decoder(z)
        return images

### 4.1 — Train the Diffusion Model

In [ ]:
# -- Training the conditional latent diffusion model --

DIFF_EPOCHS = 50

autoencoder.eval()   # frozen, but set to eval for good measure

diffusion = ConditionalLatentDiffusionModel(
    autoencoder=autoencoder,
    latent_dim=LATENT_DIM,
    emb_dim=16,
    d_model=64,
    n_heads=4,
    d_ff=64,
    num_classes=NUM_CLASSES,
    num_timesteps=200,
    num_layers=4,
).to(DEVICE)

optimizer_diff = torch.optim.Adam(
    filter(lambda p: p.requires_grad, diffusion.parameters()),  # only trainable params
    lr=1e-4,
)

for epoch in range(1, DIFF_EPOCHS + 1):
    diffusion.train()
    total_loss = 0.0
    n_batches = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        loss = diffusion.compute_loss(images, labels)

        optimizer_diff.zero_grad()
        loss.backward()
        optimizer_diff.step()

        total_loss += loss.item()
        n_batches += 1

    avg_loss = total_loss / n_batches
    print(f"Epoch {epoch}/{DIFF_EPOCHS}  |  Noise Pred Loss: {avg_loss:.6f}")

print("\nDiffusion training complete. Time for the moment of truth.")

### 4.2 — Conditional Sampling
Generate digits conditioned on each class label. If they look like digits, congratulations. If they look like modern art, well, at least you tried. Diffusion models are known to take a long time to train, you can try a higher number of epochs if you wish. 

In [ ]:
# -- Sample one image per class --
diffusion.eval()

class_labels = torch.arange(0, NUM_CLASSES, device=DEVICE)   # [0, 1, 2, ..., 9]
generated = diffusion.sample(class_labels, device=DEVICE)

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(15, 2))
for i in range(NUM_CLASSES):
    axes[i].imshow(generated[i].cpu().squeeze(), cmap="gray")
    axes[i].set_title(f"'{i}'")
    axes[i].axis("off")
plt.suptitle("Conditionally Generated Digits — DDPM Reverse Diffusion")
plt.tight_layout()
plt.show()

---
## Summary

We now have: 

1. A reusable **CNN encoder** (`DimensionReductionBlock`) and a digit **classifier**.  
2. A convolutional **autoencoder** that reuses the encoder and reconstructs MNIST images.  
3. A **Transformer layer** supporting both self-attention and cross-attention.  
4. A **conditional latent diffusion model** that generates class-conditioned digits by denoising in the autoencoder's latent space, using transformer cross-attention for conditioning.